# Kubernetes

**Kubernetes (K8s)** is a platform for running containerized applications by declaring **desired state** and letting controllers continuously reconcile the cluster to match that state.


## Goals
- Understand what Kubernetes is and what problems it solves.
- Learn the cluster architecture (control plane vs worker nodes).
- Know the most important objects: Namespaces, Pods, Deployments, Services, Ingress, ConfigMaps/Secrets, PVCs, StorageClasses, HPA, Jobs/CronJobs, StatefulSets.
- Be able to read/write basic YAML manifests.


## Prerequisites
- Containers basics (images vs containers).
- Networking basics (DNS, ports, HTTP).
- (Helpful) Linux process/filesystem basics.


## What Kubernetes is (mental model)
- A **declarative API**: you submit YAML describing what you want.
- A set of **controllers** that constantly work to make reality match the declaration.
- A **scheduler** that decides which node runs which workload.

If you remember one sentence:
> Kubernetes is a control loop that keeps your app running as described.


## Why Kubernetes is used
- **Scheduling**: place workloads on nodes based on resource needs/constraints.
- **Self-healing**: restart failed containers, replace unhealthy pods.
- **Scaling**: run multiple replicas, autoscale with HPA.
- **Service discovery & load balancing**: stable DNS/VIP via Services.
- **Safe deploys**: rolling updates/rollbacks (Deployments).
- **Portability**: same primitives across clouds and on-prem.


## Cluster architecture (high level)
### Control plane
- **API Server**: front door for all operations.
- **etcd**: key-value store for cluster state.
- **Scheduler**: assigns pods to nodes.
- **Controller Manager**: reconciliation loops (deployments, nodes, endpoints, etc.).

### Worker node
- **kubelet**: agent that runs pods on the node.
- **container runtime**: containerd/CRI-O.
- **kube-proxy / CNI**: networking plumbing (depends on distro).


## Core objects (what you create)
- **Namespace**: logical partition for resources and access control.
- **Pod**: smallest deployable unit (one or more containers).
- **Deployment**: stateless workload controller (rollouts, replicas).
- **ReplicaSet**: ensures N pod replicas exist (usually managed by Deployment).
- **Service**: stable virtual IP + DNS + load-balancing to pods.
- **Ingress**: HTTP(S) routing from outside the cluster (requires an Ingress Controller).
- **ConfigMap / Secret**: configuration and sensitive data injection.
- **PersistentVolumeClaim (PVC)**: request persistent storage.
- **StorageClass**: defines how PVC storage is provisioned (cluster-scoped).
- **HPA**: autoscale pods based on metrics.
- **StatefulSet**: stable identity + storage for stateful apps.
- **Job / CronJob**: run-to-completion tasks and scheduled tasks.


## How you use Kubernetes (workflow)
1. Write a manifest (YAML) describing desired state.
2. Apply it: `kubectl apply -f manifest.yaml`.
3. Kubernetes stores desired state in etcd and controllers reconcile.
4. Observe: `kubectl get`, `kubectl describe`, `kubectl logs`.
5. Update YAML and re-apply (or use GitOps tooling to do this automatically).
6. As manifests grow, use Kustomize/Helm (base + overlays) to avoid env drift and copy/paste.


## Pseudocode: the reconciliation loop
Kubernetes is a set of controllers doing this continuously:

```text
loop forever:
  desired = read_from_api_server()
  current = observe_cluster()
  diff = desired - current
  take_actions(diff)
```

Example: if a Deployment desires 3 pods but only 2 exist, the controller creates 1 more.


## Example: a minimal web app stack (manifests)
This is the typical shape for a stateless HTTP service:

```yaml
# 1) Deployment: run N replicas
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
        - name: web
          image: example/web:1.0.0
          ports:
            - containerPort: 8080
---
# 2) Service: stable DNS + load balance to pods
apiVersion: v1
kind: Service
metadata:
  name: web
spec:
  selector:
    app: web
  ports:
    - port: 80
      targetPort: 8080
---
# 3) Ingress: route external traffic to the service
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: web
spec:
  ingressClassName: nginx
  rules:
    - host: web.example.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: web
                port:
                  number: 80
```

Each object is explained in its own notebook under `kubernetes/`.


## Common pitfalls
- Treating Kubernetes as "just Docker": networking, storage, and rollout semantics are different.
- Missing **labels/selectors**: Services and controllers rely on them.
- No **resource requests/limits**: scheduling becomes unpredictable and autoscaling breaks.
- Default traps: deploying to the **default namespace** or relying on a default **StorageClass**.
- Using Secrets incorrectly: base64 is not encryption; control access and enable encryption-at-rest.
- Running databases as Deployments: use StatefulSets + proper storage/backups.

## Exercises
- Write a Deployment + Service manifest for a simple API.
- Add a ConfigMap for configuration and reference it from the Pod.
- Add an HPA that targets 70% CPU.

## References
- Kubernetes docs: https://kubernetes.io/docs/home/
- Kubernetes API reference: https://kubernetes.io/docs/reference/kubernetes-api/
- kubectl cheat sheet: https://kubernetes.io/docs/reference/kubectl/cheatsheet/
